# 4.14 Handling Imbalanced Data

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 4: Data Preprocessing & Exploratory Data Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-04-data-preprocessing-and-exploratory-data-analysis/04.14_handling-imbalanced-data.ipynb)

Companion code for Section 4.14 of the book. Run the cells in order from top to bottom.

**Example 4.14.1**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Create highly imbalanced dataset (99% negative, 1% positive)
X, y = make_classification(
    n_samples=10000, n_features=20, n_informative=10,
    weights=[0.99, 0.01],  # 99% class 0, 1% class 1
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Class distribution:')
unique, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f'  Class {cls}: {cnt:,} samples ({cnt/len(y_train):.1%})')

# Baseline: naive model always predicts majority class
naive_pred = np.zeros_like(y_test)   # Always predict 0
naive_acc  = (naive_pred == y_test).mean()
print(f'\nNaive (always predict 0) accuracy: {naive_acc:.1%} ← misleading!')
print('(This model never finds ANY fraud/rare events!)')

# Standard logistic regression without handling imbalance
lr_default = LogisticRegression(random_state=42, max_iter=1000)
lr_default.fit(X_train, y_train)
y_pred_default = lr_default.predict(X_test)
print(f'\nLogistic Regression (no handling):')
print(classification_report(y_test, y_pred_default, target_names=['Normal','Fraud']))

**Example 4.14.2**

In [ ]:
from imblearn.over_sampling  import SMOTE, ADASYN, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler, TomekLinks, EditedNearestNeighbours
from imblearn.combine        import SMOTETomek, SMOTEENN
from imblearn.pipeline       import Pipeline as ImbPipeline
from sklearn.metrics         import f1_score, roc_auc_score
import numpy as np

def evaluate(y_test, y_pred, y_prob, name):
    f1  = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    rec = (y_pred[y_test==1]==1).mean()  # Recall on minority class
    print(f'{name:30s}: F1={f1:.3f}  AUC={auc:.3f}  Recall_minority={rec:.3f}')

results = []

# ── 1. Random Oversampling - duplicate minority samples ───────────
ros = RandomOverSampler(random_state=42)
X_ros, y_ros = ros.fit_resample(X_train, y_train)
lr = LogisticRegression(random_state=42, max_iter=1000).fit(X_ros, y_ros)
evaluate(y_test, lr.predict(X_test), lr.predict_proba(X_test)[:,1],
        'Random Oversampling')

# ── 2. SMOTE - Synthetic Minority Oversampling Technique ──────────
# Creates synthetic samples by interpolating between real minority samples
smote = SMOTE(random_state=42, k_neighbors=5)
X_smote, y_smote = smote.fit_resample(X_train, y_train)
lr = LogisticRegression(random_state=42, max_iter=1000).fit(X_smote, y_smote)
evaluate(y_test, lr.predict(X_test), lr.predict_proba(X_test)[:,1],
        'SMOTE')

# ── 3. ADASYN - Adaptive Synthetic Sampling ───────────────────────
# Focuses synthetic generation on harder-to-learn minority samples
adasyn = ADASYN(random_state=42)
X_ada, y_ada = adasyn.fit_resample(X_train, y_train)
lr = LogisticRegression(random_state=42, max_iter=1000).fit(X_ada, y_ada)
evaluate(y_test, lr.predict(X_test), lr.predict_proba(X_test)[:,1],
        'ADASYN')

# ── 4. Random Undersampling - remove majority samples ─────────────
rus = RandomUnderSampler(random_state=42)
X_rus, y_rus = rus.fit_resample(X_train, y_train)
lr = LogisticRegression(random_state=42, max_iter=1000).fit(X_rus, y_rus)
evaluate(y_test, lr.predict(X_test), lr.predict_proba(X_test)[:,1],
        'Random Undersampling')

# ── 5. SMOTETomek - oversample + clean boundary ───────────────────
st = SMOTETomek(random_state=42)
X_st, y_st = st.fit_resample(X_train, y_train)
lr = LogisticRegression(random_state=42, max_iter=1000).fit(X_st, y_st)
evaluate(y_test, lr.predict(X_test), lr.predict_proba(X_test)[:,1],
        'SMOTETomek (combined)')

print('\nInstall imbalanced-learn: pip install imbalanced-learn')

**Example 4.14.3**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report
import numpy as np

# ── 1. Class Weights - penalise misclassifying minority class ─────
# class_weight='balanced': automatically set weights inversely proportional to frequency
lr_balanced = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr_balanced.fit(X_train, y_train)
print('Logistic Regression (class_weight=balanced):')
print(classification_report(y_test, lr_balanced.predict(X_test),
      target_names=['Normal','Fraud']))

# Manual class weights
# Fraud is 99x rarer → give it 99x more weight
class_weights = {0: 1, 1: 99}
lr_manual = LogisticRegression(class_weight=class_weights, random_state=42, max_iter=1000)
lr_manual.fit(X_train, y_train)
print('Logistic Regression (manual weights {0:1, 1:99}):')
print(classification_report(y_test, lr_manual.predict(X_test),
      target_names=['Normal','Fraud']))

# ── 2. Threshold Tuning ───────────────────────────────────────────
# Default threshold is 0.5 - adjust to trade precision vs recall
lr_probs = lr_balanced.predict_proba(X_test)[:, 1]

print('Threshold sensitivity for minority class recall:')
for threshold in [0.1, 0.2, 0.3, 0.5, 0.7]:
    y_pred_thresh = (lr_probs >= threshold).astype(int)
    minority_recall = (y_pred_thresh[y_test==1]==1).mean()
    precision = (y_test[y_pred_thresh==1]==1).mean() if y_pred_thresh.sum()>0 else 0
    f1 = 2*precision*minority_recall/(precision+minority_recall+1e-8)
    print(f'  threshold={threshold:.1f}: recall={minority_recall:.2f}  precision={precision:.2f}  F1={f1:.3f}')

# ── 3. Evaluation Metrics for Imbalanced Data ─────────────────────
from sklearn.metrics import precision_recall_curve, average_precision_score

print('\nProper metrics for imbalanced datasets:')
print('    Accuracy: misleading (99% by predicting all negative)')
print('  ✓  Precision-Recall AUC: focuses on minority class')
print('  ✓  F1 Score (or F-beta for asymmetric recall/precision tradeoff)')
print('  ✓  ROC-AUC: robust but optimistic for extreme imbalance')
print('  ✓  Cohen Kappa: accounts for class imbalance in agreement measure')
print('  ✓  MCC (Matthews Correlation Coefficient): best single metric')

from sklearn.metrics import matthews_corrcoef, cohen_kappa_score
y_pred_b = lr_balanced.predict(X_test)
print(f'\n  MCC:          {matthews_corrcoef(y_test, y_pred_b):.4f}  (range [-1,1]; 1=perfect)')
print(f'  Cohen Kappa:  {cohen_kappa_score(y_test, y_pred_b):.4f}')
print(f'  PR-AUC:       {average_precision_score(y_test, lr_probs):.4f}')